Load the dataset

In [1]:
import pandas as pd
from rags import RAG
rag = RAG()
meta_qa = pd.read_parquet("parquet_data/qa_nl.parquet")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import time

def call_rag_with_retry(query, max_retries=3, base_delay=2):
    for attempt in range(max_retries):
        try:
            return rag.StepbRAG(query)
        except Exception as e:
            if attempt == max_retries - 1:
                # give up after final attempt, log and move on
                return {"answer": None, "error": str(e)}
            wait = base_delay * (2 ** attempt)  # exponential backoff: 2s, 4s, 8s...
            print(f"Retry {attempt+1}/{max_retries} after error: {e}. Waiting {wait}s...")
            time.sleep(wait)

In [ ]:
import pandas as pd
import time
from tqdm import tqdm

CHECKPOINT_EVERY = 100
results = []

overall_start = time.perf_counter()

for i, row in enumerate(tqdm(meta_qa.itertuples(index=False), total=len(meta_qa))):
    start = time.perf_counter()
    art_ids, answer = call_rag_with_retry(row.question)
    elapsed = time.perf_counter() - start

    results.append({
        "query_id": row.id,
        "query": row.question,
        "answer": answer,
        "retrieved_ids": art_ids,
        "latency_sec": elapsed
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_parquet(f"output/stepbRAG/outputs_partial{i+1}.parquet", index=False)

total_elapsed = time.perf_counter() - overall_start
print(f"Total time: {total_elapsed:.2f}s | Avg per query: {total_elapsed/len(meta_qa):.2f}s")

pd.DataFrame(results).to_parquet("output/stepbRAG/rag_outputs.parquet", index=False)

100%|██████████| 2/2 [01:03<00:00, 31.95s/it]

Total time: 63.96s | Avg per query: 0.04s


In [5]:
outp = pd.read_parquet("output/stepbRAG/rag_outputs.parquet")
outp

,query_id,query,answer,retrieved_ids,latency_sec
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,"Kort: een dagvaarding is een formele, schrifte...","[14700, 4826, 15044, 4788, 10029, 3956, 4953, ...",35.080674
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Kort: ouderlijk gezag wordt in principe door b...,"[3791, 15319, 3790, 3797, 3812, 3803, 3535, 37...",28.761383
